# Matching Logic for SkillMatch

Find the engineers who best match what a project needs, using k-nearest-neighbours (scikit-learn).

## 1. Load the engineers

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

engineers = pd.read_csv("engineers.csv")
engineers.head()

,name,role,region,vertical,years_experience,seniority,domain,communication,timezone,stack,bandwidth
0,Allison Hill,ML Engineer,EMEA,SaaS,12,60,35,37,48,42,49
1,Noah Rhodes,Full-stack Engineer,EMEA,E-commerce,9,47,47,50,8,59,34
2,Angie Henderson,Backend Engineer,EMEA,FinTech,13,63,33,65,96,28,24
3,Daniel Wagner,Full-stack Engineer,EMEA,SaaS,16,80,43,45,51,55,39
4,Cristian Santos,Frontend Engineer,Americas,Gaming,7,45,8,65,61,93,2


## 2. The six things I match on

In [2]:
attributes = ["seniority", "domain", "communication", "timezone", "stack", "bandwidth"]
engineers[attributes].head()

,seniority,domain,communication,timezone,stack,bandwidth
0,60,35,37,48,42,49
1,47,47,50,8,59,34
2,63,33,65,96,28,24
3,80,43,45,51,55,39
4,45,8,65,61,93,2


## 3. Example project needs

The six levels a project would ask for (a senior, client-facing, modern-stack, full-time role).

In [3]:
project = {
    "seniority": 85,
    "domain": 70,
    "communication": 80,
    "timezone": 60,
    "stack": 80,
    "bandwidth": 90,
}

# put the values in the same order as the attributes list
project_vector = np.array([project[a] for a in attributes])
project_vector

array([85, 70, 80, 60, 80, 90])

## 4. Find engineers with k-NN

scikit-learn's NearestNeighbors finds the closest engineers. Euclidean = exact fit.

In [4]:
knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
knn.fit(engineers[attributes].to_numpy())

distances, indices = knn.kneighbors([project_vector])
engineers.iloc[indices[0]][["name", "role", "vertical"] + attributes]

,name,role,vertical,seniority,domain,communication,timezone,stack,bandwidth
1494,Jennifer Garcia,Backend Engineer,FinTech,81,61,73,69,79,94
1897,Lauren Haley,ML Engineer,Gaming,82,66,70,47,80,78
1458,Richard Smith,Frontend Engineer,Gaming,76,50,74,55,87,98
1934,Brenda Pierce,Full-stack Engineer,Gaming,99,51,80,53,90,96
1239,Richard Turner,Backend Engineer,Healthcare,83,71,66,52,59,96
767,Haley Johnson,Full-stack Engineer,E-commerce,85,60,77,62,52,86
1384,Jacqueline Dickerson,Full-stack Engineer,SaaS,65,74,71,61,77,69
235,Kathryn Snyder,Data Engineer,E-commerce,63,66,82,39,84,89
296,Laura Haney,Full-stack Engineer,E-commerce,68,59,79,76,62,80
164,Michael Powell,ML Engineer,FinTech,69,85,65,68,62,92


## 5. Match %

Turn the distance into an easy 0-100 score (closer = higher).

In [5]:
# the biggest possible distance (all six values 99 apart)
max_distance = np.sqrt(len(attributes) * (99 ** 2))

results = engineers.iloc[indices[0]].copy()
results["match_percent"] = ((1 - distances[0] / max_distance) * 100).round(1)
results[["name", "role", "match_percent"]]

,name,role,match_percent
1494,Jennifer Garcia,Backend Engineer,93.6
1897,Lauren Haley,ML Engineer,91.4
1458,Richard Smith,Frontend Engineer,89.4
1934,Brenda Pierce,Full-stack Engineer,88.8
1239,Richard Turner,Backend Engineer,88.8
767,Haley Johnson,Full-stack Engineer,87.5
1384,Jacqueline Dickerson,Full-stack Engineer,87.3
235,Kathryn Snyder,Data Engineer,87.2
296,Laura Haney,Full-stack Engineer,86.4
164,Michael Powell,ML Engineer,86.3


## 6. Potential fit (cosine)

Switch the metric to cosine to match the shape of the needs, not the exact levels. This finds engineers who could stretch into the role.

In [6]:
potential_knn = NearestNeighbors(n_neighbors=10, metric="cosine")
potential_knn.fit(engineers[attributes].to_numpy())

p_dist, p_idx = potential_knn.kneighbors([project_vector])
engineers.iloc[p_idx[0]][["name", "role", "vertical"] + attributes]

,name,role,vertical,seniority,domain,communication,timezone,stack,bandwidth
1897,Lauren Haley,ML Engineer,Gaming,82,66,70,47,80,78
1866,Samuel Ellis,Frontend Engineer,Gaming,71,57,57,49,57,62
1494,Jennifer Garcia,Backend Engineer,FinTech,81,61,73,69,79,94
892,Jake Shaw,Frontend Engineer,Healthcare,71,67,72,48,66,62
1811,Connor Dominguez,ML Engineer,E-commerce,53,59,56,46,61,58
59,John Daniel,Frontend Engineer,Gaming,70,54,58,51,69,56
874,Jesse Lynch,Backend Engineer,FinTech,49,49,37,34,50,57
1006,Kristen Randall,DevOps Engineer,Healthcare,68,61,56,55,50,65
1860,Jaime Walker,Full-stack Engineer,SaaS,47,40,40,42,40,42
1995,Cathy Bell,Backend Engineer,SaaS,71,51,61,49,56,89


## 7. Adding weights

Make some things matter more. Here seniority and communication count double.

In [7]:
weights = {"seniority": 2, "domain": 1, "communication": 2, "timezone": 1, "stack": 1, "bandwidth": 1}
weight_vector = np.array([weights[a] for a in attributes])

weighted_knn = NearestNeighbors(n_neighbors=10, metric="euclidean")
weighted_knn.fit(engineers[attributes].to_numpy() * weight_vector)

w_dist, w_idx = weighted_knn.kneighbors([project_vector * weight_vector])
engineers.iloc[w_idx[0]][["name", "role"] + attributes]

,name,role,seniority,domain,communication,timezone,stack,bandwidth
1494,Jennifer Garcia,Backend Engineer,81,61,73,69,79,94
1897,Lauren Haley,ML Engineer,82,66,70,47,80,78
767,Haley Johnson,Full-stack Engineer,85,60,77,62,52,86
1458,Richard Smith,Frontend Engineer,76,50,74,55,87,98
1934,Brenda Pierce,Full-stack Engineer,99,51,80,53,90,96
1239,Richard Turner,Backend Engineer,83,71,66,52,59,96
1219,Lisa Haley,Full-stack Engineer,93,66,76,39,83,57
209,Brian Lee,Full-stack Engineer,99,50,72,48,67,78
296,Laura Haney,Full-stack Engineer,68,59,79,76,62,80
1426,Robert Charles,Solutions Architect,91,56,74,22,69,90


## 8. One function for everything

Combines the metric choice and weights. I will use this in the backend next.

In [8]:
def recommend(project, method="euclidean", weights=None, top_n=10):
    if weights is None:
        weights = {a: 1 for a in attributes}
    weight_vector = np.array([weights[a] for a in attributes])

    # apply the weights to both the engineers and the project needs
    engineer_data = engineers[attributes].to_numpy() * weight_vector
    project_point = np.array([project[a] for a in attributes]) * weight_vector

    # find the nearest engineers with k-NN
    knn = NearestNeighbors(n_neighbors=top_n, metric=method)
    knn.fit(engineer_data)
    distances, indices = knn.kneighbors([project_point])

    # turn the distance into an easy 0-100 match score
    if method == "euclidean":
        max_distance = np.sqrt((weight_vector ** 2 * (99 ** 2)).sum())
        match = (1 - distances[0] / max_distance) * 100
    else:  # cosine distance is 1 - similarity
        match = (1 - distances[0]) * 100

    result = engineers.iloc[indices[0]].copy()
    result["match_percent"] = match.round(1)
    return result[["name", "role", "vertical"] + attributes + ["match_percent"]]


# test it
recommend(project, method="euclidean", top_n=5)

,name,role,vertical,seniority,domain,communication,timezone,stack,bandwidth,match_percent
1494,Jennifer Garcia,Backend Engineer,FinTech,81,61,73,69,79,94,93.6
1897,Lauren Haley,ML Engineer,Gaming,82,66,70,47,80,78,91.4
1458,Richard Smith,Frontend Engineer,Gaming,76,50,74,55,87,98,89.4
1239,Richard Turner,Backend Engineer,Healthcare,83,71,66,52,59,96,88.8
1934,Brenda Pierce,Full-stack Engineer,Gaming,99,51,80,53,90,96,88.8


## 9. Quick test

A fixed set of project needs, and the engineer names it returns.

In [9]:
test_project = {
    "seniority": 30,
    "domain": 40,
    "communication": 25,
    "timezone": 50,
    "stack": 70,
    "bandwidth": 80,
}

results = recommend(test_project, method="euclidean", top_n=5)
print(results["name"].tolist())
results

['Larry Lane', 'Donald Wright', 'Richard Stevens', 'Kiara Little', 'Dana Bennett']


,name,role,vertical,seniority,domain,communication,timezone,stack,bandwidth,match_percent
1736,Larry Lane,DevOps Engineer,E-commerce,29,40,29,33,59,88,90.9
63,Donald Wright,Data Engineer,E-commerce,26,48,38,37,60,81,90.6
831,Richard Stevens,DevOps Engineer,Healthcare,32,30,38,37,61,83,90.5
591,Kiara Little,DevOps Engineer,E-commerce,40,40,33,57,52,87,90.0
747,Dana Bennett,ML Engineer,Healthcare,35,25,36,49,81,70,90.0
